## Hydra Composition Walkthrough

This notebook is a practical guide to how Deckard uses Hydra for composition and multirun sweeps.

What you will see:

- how `defaults` picks data/model/defense/attack/score profiles,
- how Hydra defaults overrides activate the Optuna sweeper,
- how runtime CLI-style overrides modify sweeper behavior,
- where sweep/search profile files live under `examples/*/config/`.

The examples in this notebook mirror the canonical sklearn config and the integration-test composition patterns.

In [1]:
from pathlib import Path

from hydra import compose, initialize_config_dir
from hydra.core.config_store import ConfigStore
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf

PROJECT_ROOT = Path("../..").resolve()
CONFIG_DIR = PROJECT_ROOT / "examples" / "sklearn" / "config"


# This function allows us to start with a blank hydra context.
# Normally, you configure the context by setting environment variables
# or by setting up a .deckard_rc file in a target working directory.
# This is addressed in a later notebook.
def reset_hydra_state() -> None:
    if GlobalHydra.instance().is_initialized():
        GlobalHydra.instance().clear()
    config_store = ConfigStore.instance()
    for key in list(config_store.repo.keys()):
        if key not in {"hydra", "_dummy_empty_config_.yaml"}:
            config_store.repo.pop(key, None)


reset_hydra_state()

# This is the hydra python API for parsing CONFIG_DIR/default.yaml
with initialize_config_dir(version_base="1.3", config_dir=str(CONFIG_DIR)):
    default_cfg = compose(
        config_name="default",
        overrides=[
            "score=classification",
        ],
    )
# When resolve=True, hydra will replace any templated strings (syntax: ${template}).
# Resolve only the score subtree to avoid custom interpolation resolvers outside this scope.
default_score_cfg = OmegaConf.to_container(default_cfg.score, resolve=True)

print("Default score keys:", list(default_score_cfg["scorers"].keys()))

print("Data alias template:", default_cfg.data_alias)
print("Model alias template:", default_cfg.model_alias)

Default score keys: ['accuracy', 'precision', 'recall', 'f1', 'log_loss']
Data alias template: adult
Model alias template: rf


In [2]:
reset_hydra_state()
with initialize_config_dir(version_base="1.3", config_dir=str(CONFIG_DIR)):
    survival_cfg = compose(config_name="survival")
survival_score_cfg = OmegaConf.to_container(survival_cfg.score, resolve=True)
print("Survival score keys:", list(survival_score_cfg["scorers"].keys()))

Survival score keys: ['concordance', 'aic', 'bic']


## Hydra Overrides In Defaults

The canonical config in `examples/sklearn/config/default.yaml` uses Hydra defaults overrides to activate Optuna sweeps:

- `override hydra/sweeper: optuna` selects the Optuna sweeper plugin.
- `override hydra/sweeper/sampler: random` selects the random sampler profile.

Hydra does not include the full `hydra` tree in composed configs by default, so we compose with `return_hydra_config=True` when we want to inspect it directly.

In [3]:
from omegaconf import OmegaConf

reset_hydra_state()
with initialize_config_dir(version_base="1.3", config_dir=str(CONFIG_DIR)):
    default_with_hydra = compose(
        config_name="default",
        overrides=["score=classification"],
        return_hydra_config=True,
    )

default_full = OmegaConf.to_container(default_with_hydra, resolve=False)
defaults_list = default_full.get("defaults", [])

hydra_defaults = []
for entry in defaults_list:
    if isinstance(entry, str) and "hydra/" in entry:
        hydra_defaults.append(entry)
    elif isinstance(entry, dict) and any("hydra/" in key for key in entry.keys()):
        hydra_defaults.append(entry)

print("Hydra defaults overrides:")
for entry in hydra_defaults:
    print(" -", entry)

print("\nHydra summary:")
print(" sweeper target:", default_with_hydra.hydra.sweeper._target_)
print(" sampler target:", default_with_hydra.hydra.sweeper.sampler._target_)
print(" study name template:", default_with_hydra.hydra.sweeper.study_name)
print(" directions:", list(default_with_hydra.directions))
print(" optimizers:", list(default_with_hydra.optimizers))

Hydra defaults overrides:

Hydra summary:
 sweeper target: hydra_plugins.hydra_optuna_sweeper.optuna_sweeper.OptunaSweeper
 sampler target: optuna.samplers.RandomSampler
 study name template: adult_rf_class-labels_hsj
 directions: ['maximize', 'maximize', 'maximize']
 optimizers: ['accuracy', 'evasion_accuracy', 'attack_generation_time']


## Runtime Operators And sweeper.params

Hydra override operators:

- `key=value`: set/override an existing value.
- `+key=value`: add a new key (fails if key already exists).
- `++key=value`: add or override (safe for both cases).
- `~key`: remove a key or selected config group option.

Examples:

- `hydra.sweeper.n_trials=2`
- `++hydra.sweeper.study_name=notebook_demo`
- `+notes=debug_run`
- `~defense`

`hydra.sweeper.params` defines search-space expressions for the Optuna sweeper, for example:

- `++hydra.sweeper.params.model.model_params.max_depth=int(interval(2,20))`
- `++hydra.sweeper.params.attack.attack_params.eps=interval(0.01,0.3)`

The next cell composes a concise runtime override set and prints only a short summary.

In [4]:
runtime_overrides = [
    "score=classification",
    "hydra.sweeper.n_trials=2",
    "hydra.sweeper.n_jobs=1",
    "hydra.sweeper.study_name=notebook_demo",
]

reset_hydra_state()
with initialize_config_dir(version_base="1.3", config_dir=str(CONFIG_DIR)):
    runtime_cfg = compose(
        config_name="default",
        overrides=runtime_overrides,
        return_hydra_config=True,
    )

print("Applied overrides:")
for item in runtime_overrides:
    print(" -", item)

print("\nResolved sweeper block:\n")
print(OmegaConf.to_yaml(runtime_cfg.hydra.sweeper, resolve=True))

print("Resolved callback block:\n")
print(OmegaConf.to_yaml(runtime_cfg.hydra.callbacks.deckard_optuna, resolve=True))

Applied overrides:
 - score=classification
 - hydra.sweeper.n_trials=2
 - hydra.sweeper.n_jobs=1
 - hydra.sweeper.study_name=notebook_demo

Resolved sweeper block:

sampler:
  _target_: optuna.samplers.RandomSampler
  seed: 42
_target_: hydra_plugins.hydra_optuna_sweeper.optuna_sweeper.OptunaSweeper
direction:
- maximize
- maximize
- maximize
storage: sqlite:///optuna.db
study_name: notebook_demo
n_trials: 2
n_jobs: 1
max_failure_rate: 1.0
search_space: null
params:
  ++model.model_params.n_estimators: int(range(1,1000))
  ++model.model_params.max_depth: int(range(1,101))
  ++model.model_params.max_leaf_nodes: int(range(2,1000))
  ++defense.defense_name: art.defences.postprocessor.ClassLabels
  ++defense.defense_params.apply_fit: bool(False)
  ++defense.defense_params.apply_predict: bool(True,False)
  ++attack.attack_params.max_iter: int(range(10,100))
  ++attack.attack_params.max_eval: int(range(50,1000))
  ++attack.attack_params.init_eval: int(range(10,50))
  ++attack.attack_params.i

## Where `config/sweep` Maps In This Repo

In this branch, sweep profiles are organized under `examples/*/config/search/` rather than a literal `examples/*/config/sweep/` folder.

For sklearn, these groups back the defaults entries in `default.yaml`:

- `search/models`
- `search/defenses`
- `search/attacks`

Conceptually, `config/search` here serves the same role many Hydra projects call `config/sweep`: reusable search-space/profile snippets consumed during multirun optimization.

In [5]:
from pathlib import Path

repo_root = Path("../..").resolve()
examples_root = repo_root / "examples"

print("Sweep/search profile roots by example:")
for config_dir in sorted(examples_root.glob("*/config")):
    search_dir = config_dir / "search"
    sweep_dir = config_dir / "sweep"
    if search_dir.exists() or sweep_dir.exists():
        print(f" - {config_dir.relative_to(repo_root)}")
        if search_dir.exists():
            groups = sorted([p.name for p in search_dir.iterdir() if p.is_dir()])
            print(
                "    uses config/search groups:", groups if groups else "(files only)"
            )
        if sweep_dir.exists():
            groups = sorted([p.name for p in sweep_dir.iterdir() if p.is_dir()])
            print(
                "    uses config/sweep groups:", groups if groups else "(files only)"
            )

sklearn_search = repo_root / "examples" / "sklearn" / "config" / "search"
if sklearn_search.exists():
    print("\nSample sklearn search profiles:")
    for rel in [
        "models/rf.yaml",
        "models/logistic.yaml",
        "defenses/class-labels.yaml",
        "attacks/hsj.yaml",
    ]:
        path = sklearn_search / rel
        print(" -", rel, "[exists]" if path.exists() else "[missing]")

Sweep/search profile roots by example:
 - examples/sklearn/config
    uses config/search groups: ['attacks', 'defenses', 'models']

Sample sklearn search profiles:
 - models/rf.yaml [exists]
 - models/logistic.yaml [exists]
 - defenses/class-labels.yaml [exists]
 - attacks/hsj.yaml [exists]


## Single-Default Multi-Stage Execution Templates (Phase 6)

The following cells show executable command templates against `examples/sklearn/config/default.yaml`:

- single-run through selected stages
- multirun fan-out with Optuna storage overrides

These templates keep stage/trial control in runtime overrides rather than alternate orchestration code paths.

In [6]:
import shutil
from pathlib import Path

repo_root = Path("../..").resolve()
config_dir = repo_root / "examples" / "sklearn" / "config"
build_dir = repo_root / "docs" / "build" / "hydra_notebook"
build_dir.mkdir(parents=True, exist_ok=True)

deckard_cmd = shutil.which("deckard") or "deckard"
optuna_db = build_dir / "hydra_phase6.db"

single_stage_cmd = [
    deckard_cmd,
    "optimize",
    "--config-path",
    config_dir.as_posix(),
    "--config-name",
    "default",
    "score=classification",
    "stage=score",
    "hydra.sweeper.n_trials=1",
    f"+files.params_file={(build_dir / 'single_params.yaml').as_posix()}",
    f"+files.score_file={(build_dir / 'single_scores.json').as_posix()}",
]

multirun_stage_cmd = [
    deckard_cmd,
    "optimize",
    "--multirun",
    "--config-path",
    config_dir.as_posix(),
    "--config-name",
    "default",
    "score=classification",
    "stage=score,persist",
    "hydra.sweeper.n_trials=4",
    "hydra.sweeper.n_jobs=1",
    "hydra.sweeper.study_name=hydra_phase6_notebook",
    f"hydra.sweeper.storage=sqlite:///{optuna_db.as_posix()}",
]

print("Single-run command template:")
print(" ".join(single_stage_cmd))
print("\nMultirun command template:")
print(" ".join(multirun_stage_cmd))

Single-run command template:
/Users/c.meyers/.pyenv/versions/deckard/bin/deckard optimize --config-path /Users/c.meyers/Documents/deckard/examples/sklearn/config --config-name default score=classification stage=score hydra.sweeper.n_trials=1 +files.params_file=/Users/c.meyers/Documents/deckard/docs/build/hydra_notebook/single_params.yaml +files.score_file=/Users/c.meyers/Documents/deckard/docs/build/hydra_notebook/single_scores.json

Multirun command template:
/Users/c.meyers/.pyenv/versions/deckard/bin/deckard optimize --multirun --config-path /Users/c.meyers/Documents/deckard/examples/sklearn/config --config-name default score=classification stage=score,persist hydra.sweeper.n_trials=4 hydra.sweeper.n_jobs=1 hydra.sweeper.study_name=hydra_phase6_notebook hydra.sweeper.storage=sqlite:////Users/c.meyers/Documents/deckard/docs/build/hydra_notebook/hydra_phase6.db
